# OpenDC Demo 4
### Failures

In the [experiment 1](1.first_experiment.ipynb), we learned how to run OpenDC experiments and how to analyze and visualize the results to learn about the behavior of a data center. In this demo, we are using OpenDC to determine the impact of failures on the behaviour of a data center.

In this demo, we will explore how machine failure can impact the performance of a data center. We run the same workload on data centers that are experiencing different levels of machine failure. 

# Failures

Failure causes hosts to stop periodically. In OpenDC, failures can be simulated by providing a trace. 
This trace describes when failures occur, how long they last, and how severe they are (i.e., the number of hosts affected). 

In this demo, we will investigate the effect of failures.

#### Let's start by looking at one of the failure traces.

In [ ]:
import pandas as pd

df_failure = pd.read_parquet("failure_traces/Facebook_user_reported.parquet")

df_failure

- *failure_interval* determines the time between failures
- *failure_duration* determines how long a machine cannot be used
- *failure_intensity* determines the ratio of machines affected by the failure.

## Experiment

A user can activate the use of failures by adding it to the Experiment file, as shown below:

```json
{
    "name": "4.failures",
    "topologies": [
        {
            "importFrom": "topologies/4.failures/surfsara.json"
        }
    ],
    "workloads": [
        {
            "type": "trace",
            "source": {
                "type": "named",
                "name": "workload_traces/surf_week"
            }
        }
    ],
    "failureModels": [
        {
            "type": "no"
        },
        {
            "type": "traceBased",
            "source": {
                "type": "named",
                "name": "failure_traces/Facebook_user_reported.parquet"
            }
        }
    ],
    "exportModels": [
        {
            "exportInterval": "3600 s",
            "printFrequency": 24,
            "filesToExport": [
                "host",
                "powerSource",
                "service",
                "task"
            ]
        }
    ]
}
```

Failures are added using the "failureModels" parameter. In this experiment, we run two simulations. One without any failures, and one simulation in which the data center was injected with failures based on the Facebook_user_reported failure trace. 

**Exercise 1:** 
Extend the experiment file located [here](experiments/4.failures/failure_experiment.json) with more failure traces. See the [failure_traces](failure_traces) folder for all available traces.

<details>
<summary>Click to reveal the answer to Exercise 1</summary>

See the [answer experiment file](experiments/4.failures_answers/failure_experiment.json)

</details>

# Running an Experiment

An experiment can be run directly from the terminal using the OpenDCExperimentRunner.

In [ ]:
import subprocess

pathToScenario = "experiments/4.failures/failure_experiment.json"
subprocess.run(["OpenDCExperimentRunner/bin/opendc", "run", pathToScenario])

**Note:** 
Because of the failures, not all tasks are able to be completed. 
When a task fails too many times, it is terminated from the system. This will produce a warning message.

# Output

**Exercise 2:** 
Load the results into Pandas DataFrames


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Put your code here

<details>
<summary>Click to reveal the answer to Exercise 2</summary>

```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


df_host_no = pd.read_parquet("output/4.failures/raw-output/0/seed=0/host.parquet")
df_powerSource_no = pd.read_parquet("output/4.failures/raw-output/0/seed=0/powerSource.parquet")
df_task_no = pd.read_parquet("output/4.failures/raw-output/0/seed=0/task.parquet")
df_service_no = pd.read_parquet("output/4.failures/raw-output/0/seed=0/service.parquet")

df_host_facebook = pd.read_parquet("output/4.failures/raw-output/1/seed=0/host.parquet")
df_powerSource_facebook = pd.read_parquet("output/4.failures/raw-output/1/seed=0/powerSource.parquet")
df_task_facebook = pd.read_parquet("output/4.failures/raw-output/1/seed=0/task.parquet")
df_service_facebook = pd.read_parquet("output/4.failures/raw-output/1/seed=0/service.parquet")

df_host_instagram = pd.read_parquet("output/4.failures/raw-output/2/seed=0/host.parquet")
df_powerSource_instagram = pd.read_parquet("output/4.failures/raw-output/2/seed=0/powerSource.parquet")
df_task_instagram = pd.read_parquet("output/4.failures/raw-output/2/seed=0/task.parquet")
df_service_instagram = pd.read_parquet("output/4.failures/raw-output/2/seed=0/service.parquet")

df_host_netflix = pd.read_parquet("output/4.failures/raw-output/3/seed=0/host.parquet")
df_powerSource_netflix = pd.read_parquet("output/4.failures/raw-output/3/seed=0/powerSource.parquet")
df_task_netflix = pd.read_parquet("output/4.failures/raw-output/3/seed=0/task.parquet")
df_service_netflix = pd.read_parquet("output/4.failures/raw-output/3/seed=0/service.parquet")
```

</details>

# Aggregation

Lets calculate the runtime of the workload with the different failure traces.

**Exercise 3:**
Determine and print the runtime of the workload when exposed to different failure traces.

In [ ]:
# Your code goes here...

<details>
<summary>Click to reveal the answer to Exercise 3</summary>

```python
runtime_no = pd.to_timedelta(df_service_no.timestamp.max() - df_service_no.timestamp.min(), unit="ms")

runtime_facebook = pd.to_timedelta(df_service_facebook.timestamp.max() - df_service_facebook.timestamp.min(), unit="ms")
runtime_instagram = pd.to_timedelta(df_service_instagram.timestamp.max() - df_service_instagram.timestamp.min(), unit="ms")
runtime_netflix = pd.to_timedelta(df_service_netflix.timestamp.max() - df_service_netflix.timestamp.min(), unit="ms")


print(f"The workload took {runtime_no} without failures")
print(f"The workload took {runtime_facebook} with facebook failures")
print(f"The workload took {runtime_instagram} with instagram failures")
print(f"The workload took {runtime_netflix} with netflix failures")
```

</details>

## Sustainability

**Exercise 4:**
Calculate and print the total energy usage and carbon emission of the workload when exposed to different failure traces.

In [ ]:
# Your code goes here...

<details>
<summary>Click to reveal the answer to Exercise 4</summary>

```python
energy_no = df_powerSource_no.energy_usage.max() / 3_600_000
energy_facebook = df_powerSource_facebook.energy_usage.max() / 3_600_000
energy_instagram = df_powerSource_instagram.energy_usage.max() / 3_600_000
energy_netflix = df_powerSource_netflix.energy_usage.max() / 3_600_000

print(f"The data center used {energy_no} kWh energy without failures")
print(f"The data center used {energy_facebook} kWh energy with facebook failures")
print(f"The data center used {energy_instagram} kWh energy with instagram failures")
print(f"The data center used {energy_netflix} kWh energy with netflix failures")

carbon_no = df_powerSource_no.carbon_emission.max() / 1000
carbon_facebook = df_powerSource_facebook.carbon_emission.max() / 1000
carbon_instagram = df_powerSource_instagram.carbon_emission.max() / 1000
carbon_netflix = df_powerSource_netflix.carbon_emission.max() / 1000

print(f"The data center emitted {carbon_no} kg carbon without failures")
print(f"The data center emitted {carbon_facebook} kg carbon with facebook failures")
print(f"The data center emitted {carbon_instagram} kg carbon with instagram failures")
print(f"The data center emitted {carbon_netflix} kg carbon with netflix failures")
```

</details>

# Visualization
Failures interrupt the tasks that are running on a host causing them to restart on other hosts. Lets try to visualize this effect.

**Exercise 5:** 
Plot the number of tasks for each of the failure models

In [ ]:
# Put your code here

<details>
<summary>Click to reveal the answer to Exercise 5</summary>

```python
plt.plot(df_service_no.tasks_active, label="no")
plt.plot(df_service_facebook.tasks_active, label="Facebook")
plt.plot(df_service_instagram.tasks_active, label="Instagram")
plt.plot(df_service_netflix.tasks_active, label="Netflix")

plt.title("Active tasks during a workload")
plt.xlabel("time (h)")
plt.ylabel("active tasks")
plt.legend()
plt.show()
```

</details>

**Exercise 6:** 
Plot the energy usage of the datacenter for each of the failure models

In [ ]:
# Put your code here

<details>
<summary>Click to reveal the answer to Exercise 6</summary>

```python
plt.plot(df_powerSource_no.energy_usage, label="no")
plt.plot(df_powerSource_facebook.energy_usage, label="Facebook")
plt.plot(df_powerSource_instagram.energy_usage, label="Instagram")
plt.plot(df_powerSource_netflix.energy_usage, label="Netflix")

plt.title("Energy usage during a workload")
plt.xlabel("time [h]")
plt.ylabel("Energy Usage [J]")
plt.legend()
plt.show()
```

</details>